# TalentDesk, Section 2 Lab (Exercise): The Agentic Loop and Sessions

A hands-on exercise built on the **base Anthropic SDK** (for the hand-written loop, where
`stop_reason` is fully visible) and the **Claude Agent SDK** (for sessions). It combines the
two Section 2 skills: driving an agent with the **agentic loop** so it takes as many tool
steps as the request needs (Lab 1), and persisting a conversation so you can **resume** it
later or **fork** it for a parallel attempt (Lab 2). You fill in four short `TODO` blocks;
everything else is provided. Offline mocks let you test the loop and the resume/fork behaviour
without a key or Node.js, and a full solution is at the end. Runs **Sonnet**
(`claude-sonnet-4-6`).

## The real-world scenario

Priya's recruiter agent from Section 1 could take **one** action per request. But real work is
multi-step: *"check candidate C2's stage, then advance them"* needs two tool calls in order,
and *"triage this candidate"* might need more, with no way to know the count in advance. A
**loop** solves this: keep going while the model asks for tools, stop when it says it is done.
The signal that tells you which is `stop_reason`.

And a candidate investigation rarely finishes in one sitting. Priya closes her laptop, comes
back tomorrow, and does not want to re-explain which candidate she was chasing. **Sessions**
solve this: the conversation is saved, so she can **resume** exactly where she left off, or
**fork** it to try an alternative (draft an offer on one branch, a rejection on another)
without losing her place.

The question this lab answers: **how do you drive an agent by the model's own signals instead
of a fixed number of steps, and how do you continue and branch that work across sittings?**

## Objectives

- Implement the **agentic loop**: iterate while `stop_reason == "tool_use"`, running the
  requested tools and appending their results; stop cleanly on `end_turn`. The safety guard is
  a net, not the exit condition.
- Contrast a **model-driven** loop with a **rule-based** hardcoded workflow, and see why the
  hardcoded one cannot match the work to the request.
- Persist a conversation with the Agent SDK: capture a **session id**, **resume** to keep full
  context, and **fork** to branch safely while the original stays intact.

## The outcome you should reach

By the end you will have:

- a working loop that takes exactly the steps a request needs (one tool for a status question,
  two for a check-then-advance request) and stops on `end_turn`;
- a side-by-side showing the rule-based workflow acting even when the request did not ask for it;
- a session helper that captures the session id, a **resumed** session that resolves a
  back-reference a **fresh** session cannot, and a **fork** that gets a new id while the original
  is untouched.

Target time: **20 to 30 minutes.** Four small `TODO` blocks. The loop and the session behaviour
are both testable offline via built-in mocks; the live paths need a real key (and Node.js 18+
for the Agent SDK).

## How to run

Run top to bottom. The loop, the rule-based workflow, and the session logic are all testable
offline: an offline **mock client** drives the loop, and an offline **mock session engine**
reproduces resume and fork. To run live, paste a real key into **Setup 2/3** and re-run from the
top; the Agent SDK session cells also need **Node.js 18+**. Live model choices are not perfectly
deterministic, so treat each live run as an observation.

## 0. Setup

**This cell:** installs the packages. We install the **Agent SDK** (for sessions) and the
**base Anthropic SDK** (for the hand-written loop where `stop_reason` is visible). The Agent SDK
also needs Node.js 18+, which cannot be pip-installed; the offline mocks do not need it.

In [ ]:
# ===== SETUP 1/3 - install both SDKs =====
%pip install -q claude-agent-sdk anthropic python-dotenv

**This cell:** imports what we need, pins the model, and sets a `RUN_LIVE` switch so live
calls fire only with a real key. It also imports the async and threading helpers the session
section uses.

In [ ]:
# ===== SETUP 2/3 - imports, the model, and a live/offline switch =====
import os                                       # read the API key from the environment
import sys                                      # detect Windows (it needs a special event loop)
import re                                       # pull candidate ids out of prompts (offline mocks)
import json                                     # print payloads readably
import asyncio                                  # the Agent SDK is async; we drive it ourselves
import threading                                # run that async loop in a side thread (notebook-safe)
import anthropic                                # the base Anthropic SDK (synchronous)

try:                                            # load a .env file if present
    from dotenv import load_dotenv             #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model every call will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key
print("live model calls:", "ON" if RUN_LIVE else "OFF (mocks drive the loop and sessions)")

**This cell:** the shared **TalentDesk world** and its two tools, carried over from
Section 1 (provided). Each candidate has a pipeline stage and a `cleared` flag. The loop calls
these tools; the guarded `advance_candidate` only moves a cleared candidate.

In [ ]:
# ===== SETUP 3/3 - the shared ATS and the two tools (provided) =====
CANDIDATES = {                                   # our tiny applicant tracking system
    "C1": {"stage": 3, "cleared": True},         #   at interview, cleared to advance
    "C2": {"stage": 2, "cleared": True},          #   in screening, cleared (so it CAN advance)
}
STAGE_NAMES = {1: "applied", 2: "screening", 3: "interview", 4: "offer"}   # code -> human word

def get_candidate_stage(candidate_id):            # tool 1: read the pipeline stage
    c = CANDIDATES.get(candidate_id)
    return STAGE_NAMES[c["stage"]] if c else "unknown candidate"

def advance_candidate(candidate_id):              # tool 2: the guarded action
    c = CANDIDATES.get(candidate_id)
    if not c:              return "unknown candidate"
    if not c["cleared"]:   return "blocked: screening not cleared"
    if c["stage"] == 4:    return "already at final stage (offer)"
    return "advanced to " + STAGE_NAMES[c["stage"] + 1]

RUN_TOOL = {"get_candidate_stage": get_candidate_stage, "advance_candidate": advance_candidate}
print("tools:", list(RUN_TOOL))

**This cell:** the **tool schemas** the model reads each turn (provided). Same two tools,
each taking one `candidate_id`.

In [ ]:
# ===== the tool schemas (provided) =====
_arg = {"type": "object",
        "properties": {"candidate_id": {"type": "string"}},
        "required": ["candidate_id"]}
TOOLS = [
    {"name": "get_candidate_stage",
     "description": "Look up which stage a candidate is at in the hiring pipeline.",
     "input_schema": _arg},
    {"name": "advance_candidate",
     "description": "Advance a candidate to the next stage if they are cleared.",
     "input_schema": _arg},
]
print("schemas ready:", [t["name"] for t in TOOLS])

### How the agentic loop works

A single round-trip (Section 1) handles one tool call. The **loop** repeats it: on
`stop_reason == "tool_use"`, run the requested tools, **append** their results to the
conversation, and go again; on `end_turn`, stop and return the answer. A `range()` guard is only
a safety net so a runaway never loops forever; it is **not** the exit condition. The model
decides how many steps the request needs; your code decides when to stop by reading
`stop_reason`.

**This cell:** the **offline loop mock** (provided). Offline, the loop talks to this
instead of the real API. It scripts a faithful sequence by reading the request and counting how
many tool results it has already seen: for a status-only request it asks for the stage tool once
then finishes; for a check-then-advance request it asks for the stage tool, then the advance
tool, then finishes.

In [ ]:
# ===== the offline loop mock (provided) - scripts a faithful stop_reason sequence =====
class _Blk:
    def __init__(self, type, **kw):
        self.type = type
        for k, v in kw.items():
            setattr(self, k, v)

class _Resp:
    def __init__(self, stop_reason, content):
        self.stop_reason, self.content = stop_reason, content

class _LoopMsgs:
    def create(self, **kw):
        msgs = kw["messages"]
        question = msgs[0]["content"].lower()          # the original request
        n_results = sum(                               # how many tool_results handed back so far
            1 for m in msgs
            if m["role"] == "user" and isinstance(m["content"], list)
            and any(isinstance(b, dict) and b.get("type") == "tool_result" for b in m["content"]))
        wants_advance = any(w in question for w in ("advance", "next round", "promote"))
        found = re.search(r"C\d+", msgs[0]["content"])
        cid = found.group(0) if found else "C1"
        if n_results == 0:                             # step 1: always check the stage first
            return _Resp("tool_use", [_Blk("tool_use", id="t1",
                         name="get_candidate_stage", input={"candidate_id": cid})])
        if n_results == 1 and wants_advance:           # step 2: only if the request asked to advance
            return _Resp("tool_use", [_Blk("tool_use", id="t2",
                         name="advance_candidate", input={"candidate_id": cid})])
        return _Resp("end_turn", [_Blk("text",         # otherwise: done
                     text=f"Done. (offline canned reply for {cid})")])

class LoopClient:
    def __init__(self): self.messages = _LoopMsgs()

LOOP_CLIENT = anthropic.Anthropic() if RUN_LIVE else LoopClient()   # real live, mock offline
print("loop client:", "real Anthropic" if RUN_LIVE else "offline LoopClient")

---

### 🎯 Part A - the agentic loop

**TODO 1 (about 7 minutes).** Complete `run_loop()`. The structure is provided; fill the three
marked steps:

- STEP A: return the final text when `stop_reason == "end_turn"`.
- STEP B: for every `tool_use` block this turn, run the matching tool via `RUN_TOOL` and collect a
  `tool_result` (with the block's `id` as `tool_use_id`).
- STEP C: append the collected results as one user turn, so the model sees them next iteration.

The `for step in range(max_guard)` is a safety net, not the exit rule; `stop_reason` is what ends
the loop.

In [ ]:
# ===== TODO 1 - the model-driven loop, driven only by stop_reason =====
def run_loop(question, max_guard=8):              # run the agent until the model says it is done
    messages = [{"role": "user", "content": question}]   # the running conversation
    for step in range(max_guard):                 # a generous SAFETY NET, not the exit rule
        r = LOOP_CLIENT.messages.create(model=MODEL, max_tokens=512,   # one model turn
                                        tools=TOOLS, messages=messages)
        print(f"step {step}: stop_reason =", r.stop_reason)   # THE signal we branch on

        # 👉 TODO 1a (STEP A): if r.stop_reason == "end_turn", return the joined text blocks
        #    return "".join(b.text for b in r.content if b.type == "text")

        messages.append({"role": "assistant", "content": r.content})   # keep the model's turn
        results = []                              # collect this turn's tool results
        for b in r.content:                       # walk the blocks the model produced
            if b.type == "tool_use":              #   it asked for a tool
                # 👉 TODO 1b (STEP B): run RUN_TOOL[b.name](**b.input), print it, and append a
                #    {"type":"tool_result","tool_use_id":b.id,"content":out} dict to results
                pass                              #   replace this line

        # 👉 TODO 1c (STEP C): append results to messages as one {"role":"user","content":results} turn
        pass                                       # replace this line

    return "(stopped: hit the safety guard)"      # only reached if the guard trips

**Self-check (offline).** Runs the loop on a two-step request. On the mock you should see
`get_candidate_stage`, then `advance_candidate`, then `end_turn`.

In [ ]:
# ===== self-check for TODO 1 (runs offline on the mock) =====
ans = run_loop("Check candidate C2's stage, then advance them.")
print("ANSWER:", ans)
assert "guard" not in ans, "the loop never reached end_turn - check STEP A and STEP C"
print("TODO 1 check passed" if "Done" in ans or RUN_LIVE else "review the steps")

**TODO 2 (about 4 minutes).** Write the **rule-based** contrast, `run_rulebased()`: a fixed
workflow that always checks the stage and then always advances, in that order, no matter the
request. This is pure Python (no model). It exists to show the cost of hardcoding control flow.

In [ ]:
# ===== TODO 2 - the rule-based (hardcoded) alternative =====
def run_rulebased(candidate_id):                  # a FIXED workflow: no model decides control flow
    # 👉 TODO 2a: always look up the stage
    # 👉 TODO 2b: always attempt to advance
    # 👉 TODO 2c: return a summary string like "stage=<...>; advance=<...>"
    return "TODO: implement run_rulebased"

**This cell:** the comparison on a **status-only** request. The rule-based version advances
C1 anyway (nobody asked), while the model-driven loop calls only the stage tool and stops. That is
the cost of hardcoding: it cannot match the work to the request.

In [ ]:
# ===== compare on a status-only request: "Where is C1?" =====
print("rule-based (advances too):", run_rulebased("C1"))     # runs offline; note the unwanted advance
print()
print("model-driven (does only what is asked):")
print("ANSWER:", run_loop("Where is candidate C1 right now?"))   # mock: one stage step, then end_turn

**Note on the Agent SDK.** The Agent SDK's `query()` is exactly this loop, already built:
it iterates internally until the model stops, so in practice you reach for `query()` rather than
hand-writing the loop. You will use `query()` for sessions in Part B.

---

### 🎯 Part B - sessions: resume and fork

A **session** is the conversation history saved to disk as you work. Because it persists, you can
return to it later with full context.

- **Continue** picks up the most recent session in the current directory.
- **Resume** picks up a specific session by id or name (track the id when you have several).
- **Fork** makes a *new* session starting from a copy of the original's history, then diverges;
  the original stays intact for a safe parallel attempt.

**Claude Code CLI (reference, run in a terminal inside your Git repo).** These are terminal
commands, shown for reference, not run by the notebook:

```bash
# start a NEW named session in your repo (run from the project directory)
claude --name talentdesk-C1-review

# later: continue the MOST RECENT session here, or resume a SPECIFIC one
claude --continue
claude --resume talentdesk-C1-review

# branch into a NEW session id, leaving the original intact
claude --resume talentdesk-C1-review --fork-session
```

Sessions are stored per project directory (under `~/.claude/projects/`), so resume from the same
folder you started in. In the Agent SDK the same ideas are the options `resume=session_id` and
`fork_session=True`, which you will set below.

**This cell:** the **run_async** helper (provided): a notebook-safe wrapper that runs any
async call (the real Agent SDK, or the offline mock) in its own thread and event loop, so the
session cells can be called like ordinary functions.

In [ ]:
# ===== a notebook-safe runner for async calls (provided) =====
def run_async(make_coro):                         # make_coro: a function returning a coroutine
    box = {}
    def worker():
        if sys.platform == "win32":
            loop = asyncio.ProactorEventLoop()
        else:
            loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            box["value"] = loop.run_until_complete(make_coro())
        except Exception as e:
            box["error"] = e
        finally:
            loop.close()
    t = threading.Thread(target=worker); t.start(); t.join()
    if "error" in box:
        raise box["error"]
    return box.get("value")

**This cell:** binds the session pieces (provided). Live, it imports them from the real
Agent SDK. Offline, it defines a **mock session engine** that stores each session's history in
memory and reproduces resume (same id, keeps context), fresh (new id, empty), and fork (new id,
copied history, original intact), so you can test your code without a key or Node.js.

In [ ]:
# ===== bind the Agent SDK pieces: real when live, mock when offline (provided) =====
if RUN_LIVE:
    from claude_agent_sdk import (query, ClaudeAgentOptions,
                                  AssistantMessage, ResultMessage, TextBlock)
else:
    import itertools
    _MOCK_STORE = {}                               # session_id -> the candidate it remembers
    _sid_seq = itertools.count(1)

    class TextBlock:
        def __init__(self, text): self.text = text
    class AssistantMessage:
        def __init__(self, content): self.content = content
    class ResultMessage:
        def __init__(self, session_id): self.session_id = session_id
    class ClaudeAgentOptions:
        def __init__(self, model=None, system_prompt=None, resume=None,
                     fork_session=False, continue_conversation=False, **kw):
            self.model = model; self.system_prompt = system_prompt
            self.resume = resume; self.fork_session = fork_session
            self.continue_conversation = continue_conversation

    async def query(prompt, options):              # a faithful stand-in for the SDK's query()
        if options.resume and options.fork_session:        # FORK: new id, copy history
            sid = f"sess-{next(_sid_seq)}"
            _MOCK_STORE[sid] = _MOCK_STORE.get(options.resume)
        elif options.resume:                                # RESUME: same id, keep history
            sid = options.resume
        else:                                               # FRESH: new id, empty history
            sid = f"sess-{next(_sid_seq)}"
        found = re.search(r"C\d+", prompt)                 # remember a candidate if named
        if found:
            _MOCK_STORE[sid] = found.group(0)
        remembered = _MOCK_STORE.get(sid)
        low = prompt.lower()
        if "same" in low or "that candidate" in low or "that order" in low:
            text = (f"You were reviewing candidate {remembered}, now at the interview stage."
                    if remembered else "Which candidate do you mean? I have no context.")
        elif "draft" in low or "goodwill" in low or "message" in low:
            text = f"[draft] A warm note regarding candidate {remembered or 'the candidate'}."
        else:
            text = f"Noted: now tracking candidate {remembered or '(none)'}."
        yield AssistantMessage([TextBlock(text)])
        yield ResultMessage(sid)

print("session pieces bound:", "real Agent SDK" if RUN_LIVE else "offline mock engine")

**TODO 3 (about 5 minutes).** Complete `ask()`, which runs one turn and returns
`(session_id, answer_text)`. Stream the messages: keep the latest `TextBlock` text from each
`AssistantMessage`, and read `session_id` off the `ResultMessage` (both `resume` and
`fork_session` need that id).

In [ ]:
# ===== TODO 3 - a one-turn helper that also captures the session id =====
async def ask(prompt, options):                    # run one turn -> (session_id, answer_text)
    sid, answer = None, ""
    async for message in query(prompt=prompt, options=options):   # stream the turn
        if isinstance(message, AssistantMessage):
            for block in message.content:
                # 👉 TODO 3a: if block is a TextBlock, set answer = block.text
                pass                               #   replace this line
        elif isinstance(message, ResultMessage):
            # 👉 TODO 3b: capture the session id -> sid = message.session_id
            pass                                   #   replace this line
    return sid, answer

**This cell:** **turn 1** establishes context (provided): it tells the agent we are
reviewing candidate C1, and captures the returned session id, the handle we resume and fork from.
Runs on the mock offline, the real SDK live.

In [ ]:
# ===== turn 1: establish context and capture the session id =====
BASE = ClaudeAgentOptions(
    model=MODEL,
    system_prompt="You are TalentDesk. Track the candidate the recruiter is reviewing.")

sid, ans = run_async(lambda: ask(
    "I am reviewing candidate C1, currently at interview. Note that for me.", BASE))
print("session id:", sid)
print("turn 1 answer:", ans)

**TODO 4 (about 4 minutes).** Build the two option objects that continue the thread:

- `RESUMED`: same session, by id. Set `resume=sid`.
- `FORKED`: branch off the same session. Set `resume=sid` **and** `fork_session=True`.

`BASE` above (a fresh session, no resume) is provided for the comparison.

In [ ]:
# ===== TODO 4 - resume and fork options =====
# 👉 TODO 4a: RESUMED continues the SAME session by id
RESUMED = ClaudeAgentOptions(model=MODEL)          # add: resume=sid

# 👉 TODO 4b: FORKED branches into a NEW session, leaving the original intact
FORKED = ClaudeAgentOptions(model=MODEL)           # add: resume=sid, fork_session=True

print("RESUMED.resume =", getattr(RESUMED, "resume", None))
print("FORKED.fork_session =", getattr(FORKED, "fork_session", None))

**This cell:** the payoff. A **resumed** session resolves the back-reference (it still
knows we mean C1); a **fresh** session cannot; a **fork** gets a new id with copied history while
the original id stays valid. Runs on the mock offline, the real SDK live.

In [ ]:
# ===== resume vs fresh vs fork =====
# resume: context carries over
_, a_resumed = run_async(lambda: ask("What was the stage of that same candidate?", RESUMED))
print("resumed answer:", a_resumed)

# fresh: no history, so the back-reference fails
_, a_fresh = run_async(lambda: ask("What was the stage of that same candidate?", BASE))
print("fresh answer  :", a_fresh)

# fork: a NEW id; the original sid is untouched
fork_sid, a_fork = run_async(lambda: ask(
    "On a separate branch: draft a goodwill message about that candidate.", FORKED))
print("fork session id (new):", fork_sid, "| original still:", sid)
print("fork answer:", a_fork)

---

### Anti-patterns to avoid

| anti-pattern | what to do instead |
|---|---|
| cap the loop at `range(3)` and hope | let `end_turn` end it; keep a range only as a safety net |
| scan the reply for words like "done" | branch on `stop_reason`, never on prose |
| ignore `stop_reason` and always re-ask | append every `tool_result`, then loop |
| hardcode the sequence of tool calls | let the model choose the steps the request needs |
| leave sessions unnamed and hunt later | name them at the start (`--name`, or `/rename`) |
| resume when you meant to branch | add `--fork-session` (or `fork_session=True`) to keep the original |
| rely on `--continue` in a script | capture and resume an explicit session id for reliable automation |

**Lesson:** the model decides *how many* steps the work needs; your code decides *when to
stop* by reading `stop_reason`. A rule-based workflow cannot adapt; a model-driven loop does
exactly the work the request calls for, and the Agent SDK's `query()` is that loop already built.
A **session** is that conversation, saved: **resume** to keep full context, start **fresh** for an
empty one, and **fork** to explore an alternative without risking the original.

---

## Recap - the loop and sessions

| Concept | In this lab | Course topic |
|---|---|---|
| Agentic loop | continue on `tool_use`, stop on `end_turn` | model-driven control flow (Lab 1) |
| Rule-based workflow | fixed steps, advances even when not asked | why hardcoding cannot adapt (Lab 1) |
| Agent SDK `query()` | the same loop, productised | reach for it in practice (Lab 1) |
| Resume | `resume=session_id` recalls prior context | continue work across sittings (Lab 2) |
| Fresh | a new options object, empty history | the contrast that shows resume's value (Lab 2) |
| Fork | `resume=sid, fork_session=True`, new id | branch safely, original intact (Lab 2) |

**Try it next:** give the loop a three-step request and watch it take exactly three tool turns
before `end_turn`. Then, live in a terminal, name a session with `claude --name`, resume it
tomorrow with `claude --resume`, and fork it with `--fork-session` before a risky change.